# Fine-tune one model

This notebook fine-tunes one model with one seed, evaluates the adapter on all splits with its own vLLM server, and prints the gate against the few-shot baseline of the same model. It has no logic of its own: every step calls `make`. To train another model or seed, change `MODEL` or `SEED` and run it again.

On Kaggle, use GPU T4 x2 and turn Internet on. Training runs on both GPUs (`NUM_GPUS = 2`). The evaluation starts one vLLM server on each GPU (`SERVERS = 2`) and shares the documents between them. With `DRY_RUN = True` the notebook runs without a GPU. It skips the install and the clone, uses the checkout it is in, trains a tiny random model for two steps, evaluates against a fake server instead of vLLM, and writes to a temporary folder. A dry run uses a fake model, so its results say nothing about a real model. Never copy them into `results/`.

The cells run in order. Run them all in the same session, because the later cells need what the earlier ones installed.

In [ ]:
DRY_RUN = False
REPO_URL = "https://github.com/mgetech/open-llm-adaptation.git"
REPO_REF = "main"  # a tag or a commit
MODEL = "qwen3-4b"
SEED = 0
CHECK_MODEL = "qwen3-1.7b"
CHECK_STEPS = 3
NUM_GPUS = 2  # 1 for a single GPU
SERVERS = 2  # vLLM servers for the evaluation, one per GPU

GPUS = [] if DRY_RUN else [f"NUM_GPUS={NUM_GPUS}"]
DRY = ["DRY_RUN=1"] if DRY_RUN else []
OFFLINE = ["OFFLINE=1"] if DRY_RUN else []
SMOKE = ["SMOKE=1"] if DRY_RUN else []

In [ ]:
import os
import subprocess
import tempfile
from pathlib import Path


def run(*command, check=True):
    options = dict(stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with subprocess.Popen(command, encoding="utf-8", errors="replace", **options) as process:
        for line in process.stdout:
            print(line, end="")
    if check and process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


def make(*arguments, check=True):
    run("make", *arguments, check=check)


if DRY_RUN:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    WORK = Path(tempfile.mkdtemp(prefix="training-dry-run-"))
    ADAPTERS_DIR = WORK / "adapters"
    RUNS_DIR = WORK / "runs"
    OUT_DIR = WORK / "eval"
    RUN = f"smoke-{SEED}"
    SYSTEM = f"smoke-sft-{SEED}"
    BASELINE = []
else:
    REPO = Path("/kaggle/working/open-llm-adaptation")
    ADAPTERS_DIR = REPO / "adapters"
    RUNS_DIR = REPO / "results" / "runs"
    OUT_DIR = REPO / "results" / "eval"
    RUN = f"{MODEL}-{SEED}"
    SYSTEM = f"{MODEL}-sft-{SEED}"
    BASELINE = [f"BASELINE={MODEL}-fewshot"]
print("repository:", REPO)
print("adapters:", ADAPTERS_DIR)
print("run manifests:", RUNS_DIR)
print("results:", OUT_DIR)

## Python 3.12

Kaggle's default Python may be older, so uv installs its own 3.12.

In [ ]:
if DRY_RUN:
    print("dry run: skipped")
else:
    run("pip", "install", "--quiet", "uv")
    run("uv", "python", "install", "3.12")

## Repository

The data is in the repository, so nothing else is downloaded. If the repository is already there, for example after a restart of this cell, it is updated and reused.

In [ ]:
if DRY_RUN:
    print("dry run: skipped")
else:
    if not REPO.exists():
        run("git", "clone", REPO_URL, str(REPO))
    run("git", "-C", str(REPO), "fetch", "--tags", "origin")
    run("git", "-C", str(REPO), "checkout", REPO_REF)
os.chdir(REPO)

## Project and vLLM

The project is installed from the lock file. vLLM is the version pinned in `configs/models.yaml`. It needs the same torch version as the training libraries, so one environment serves both. The training libraries are installed by the first `make train`.

In [ ]:
if DRY_RUN:
    print("dry run: skipped")
else:
    make("setup")
    make("install-vllm")

## Model checks

The licence, size and serving checks, and the pinned revisions against the Hub.

In [ ]:
make("models-check", *OFFLINE)

## Short check (optional, about 15 minutes)

Before the long run, this trains the small model for a few steps and evaluates the dev split with that adapter. It uses the same number of GPUs as the long run, so it shows early whether the GPUs, the 4-bit loading and the serving of an adapter work, and it installs the training libraries. Its names end in `-steps`, the number of steps, `-gpus` and the number of GPUs, so it never replaces a real run. Do not copy its files into `results/`.

In [ ]:
make(
    "train",
    f"MODEL={CHECK_MODEL}",
    f"SEED={SEED}",
    f"MAX_STEPS={CHECK_STEPS}",
    f"ADAPTERS_DIR={ADAPTERS_DIR.as_posix()}",
    f"RUNS_DIR={RUNS_DIR.as_posix()}",
    *GPUS,
    *SMOKE,
)

In [ ]:
make(
    "eval-sft",
    f"MODEL={CHECK_MODEL}",
    f"SEED={SEED}",
    f"MAX_STEPS={CHECK_STEPS}",
    *GPUS,
    f"SERVERS={SERVERS}",
    "SPLITS=dev",
    f"ADAPTERS_DIR={ADAPTERS_DIR.as_posix()}",
    f"OUT_DIR={OUT_DIR.as_posix()}",
    *DRY,
    *SMOKE,
)

## Training

QLoRA fine-tuning with the settings of `configs/train.yaml`. With `NUM_GPUS = 2`, each GPU holds the whole 4-bit model and trains on half of every batch, and the effective batch size stays 16. A checkpoint is written every 100 steps. If the session stops in the middle, run the cells again up to this one and add `RESUME=1` to the call below to continue from the last checkpoint.

In [ ]:
make(
    "train",
    f"MODEL={MODEL}",
    f"SEED={SEED}",
    f"ADAPTERS_DIR={ADAPTERS_DIR.as_posix()}",
    f"RUNS_DIR={RUNS_DIR.as_posix()}",
    *GPUS,
    *SMOKE,
)

## Evaluation

`make eval-sft` starts the servers with the base model and the adapter, one per GPU. It runs dev, test_iid, test_ood, edge and gold with the prompt the adapter was trained on, shares the documents of each split between the servers, and stops the servers at the end. With `SERVERS = 1` it uses one server on the first GPU.

In [ ]:
make(
    "eval-sft",
    f"MODEL={MODEL}",
    f"SEED={SEED}",
    f"SERVERS={SERVERS}",
    f"ADAPTERS_DIR={ADAPTERS_DIR.as_posix()}",
    f"OUT_DIR={OUT_DIR.as_posix()}",
    *DRY,
    *SMOKE,
)

## Gate

The table compares the adapter with the few-shot baseline of the same model, which is in the repository. A model that does not pass the gate is not shipped.

In [ ]:
make("gate", f"SYSTEM={SYSTEM}", *BASELINE, f"OUT_DIR={OUT_DIR.as_posix()}", check=False)

## Results

Copy the run manifest from `results/runs/` and the folder of the system from `results/eval/tax-booking/` into the same places in the repository. Do not copy files that have `-steps` in their name. The adapter stays out of the repository. To download all of it from the Output tab, zip it first, in a cell that starts with `!`:

```
cd /kaggle/working/open-llm-adaptation && zip -r /kaggle/working/results.zip results adapters/tax-booking
```

In [ ]:
for path in sorted(RUNS_DIR.glob("*.json")):
    print(path)
for path in sorted(OUT_DIR.rglob("*.json")):
    print(path)
for path in sorted((ADAPTERS_DIR / "tax-booking" / RUN).glob("*")):
    print(path)